In [1]:
EXPERIMENTAL_ID="product_search_custom_scoring"
START_DATE="2025-09-12"

In [2]:
from odps_client import get_odps_sql_result_as_df
from datetime import datetime, timedelta

daily_high_search_volume_theshold = 470 / 14
daily_low_search_volume_theshold = 2

last_n_days = 60
ds_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
last_n_days_ago = (datetime.now() - timedelta(days=last_n_days)).strftime("%Y%m%d")

        #         ,CASE
        #     WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > {last_n_days*daily_high_search_volume_theshold} THEN '高频搜索词'
        #     WHEN COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) <= {last_n_days*daily_low_search_volume_theshold} THEN '低频搜索词'
        #     ELSE '中频搜索词'
        #  END

top_query = f"""
SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS std_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.5) AS p50_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.75) AS p75_click_index
        ,PERCENTILE(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END,0.9) AS p90_click_index
        ,RANK() OVER (PARTITION BY "dontcarte" ORDER BY COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) DESC) as search_cnt_rnk
        ,"不区分频次" AS 搜索频次标签
        ,COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) as ctr_uv
        ,COUNT(distinct ds) as 有搜索天数
        ,CASE
            WHEN COUNT(DISTINCT CASE WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) * 1.0 / COUNT(DISTINCT CASE WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) > 0.25 THEN '高点击率词'
            ELSE '低点击率词'
         END AS 点击率标签
FROM    summerfarm_tech.app_log_search_detail_di
WHERE   ds BETWEEN '{last_n_days_ago}' and '{ds_yesterday}'
GROUP BY query
ORDER BY searched_users DESC;
"""

top_query_df = get_odps_sql_result_as_df(sql=top_query)
top_query_df['搜索频次标签']=top_query_df['search_cnt_rnk'].apply(lambda x: 'top400' if x <= 400 else 'top400以外')
top_query_df.head(20)

2025-09-15 14:16:09 - INFO - Thread count: 20
2025-09-15 14:16:34 - INFO - Tunnel session created: <InstanceDownloadSession id=2025091514163355203b1a20248d28 project_name=summerfarm_ds instance_id=20250915061609633g5q1c9az9vt3>
2025-09-15 14:16:35 - INFO - sql:

SELECT  query
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN cust_id END) AS searched_users
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'impression' THEN CONCAT(time,cust_id) END) AS search_cnt
        ,COUNT(DISTINCT CASE    WHEN envent_type = 'click' THEN CONCAT(time,cust_id) END) AS click_cnt
        ,ROUND(AVG(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS avg_click_index
        ,ROUND(MAX(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS max_click_index
        ,ROUND(MIN(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS min_click_index
        ,ROUND(STDDEV(CASE    WHEN envent_type = 'click' THEN CAST(idx AS BIGINT) END),1) AS 

,query,searched_users,search_cnt,click_cnt,avg_click_index,max_click_index,min_click_index,std_click_index,p50_click_index,p75_click_index,p90_click_index,search_cnt_rnk,搜索频次标签,ctr_uv,有搜索天数,点击率标签
0,芒果,12611,246988,77566,5.7,178.0,0.0,7.2,3.0,7.0,17.0,1,top400,0.314048,60,高点击率词
1,牛奶,11978,106297,40909,4.7,199.0,0.0,9.3,1.0,5.0,18.0,4,top400,0.384856,60,高点击率词
2,柠檬,9960,109439,37103,5.2,554.0,0.0,8.9,3.0,7.0,12.0,2,top400,0.339029,60,高点击率词
3,草莓,8614,106951,38720,2.8,392.0,0.0,6.2,1.0,4.0,7.0,3,top400,0.362035,60,高点击率词
4,蓝莓,8414,103822,34795,1.9,274.0,0.0,4.6,1.0,2.0,4.0,5,top400,0.335141,60,高点击率词
5,安佳,8076,60358,20041,2.4,211.0,0.0,6.5,0.0,2.0,9.0,8,top400,0.332036,60,高点击率词
6,奶油,7574,71919,18701,15.4,284.0,0.0,21.3,6.0,22.0,43.0,7,top400,0.260029,60,高点击率词
7,黄油,6390,47713,10936,11.0,206.0,0.0,14.0,6.0,18.0,28.0,9,top400,0.229204,60,低点击率词
8,西瓜,5442,86363,26668,3.7,86.0,0.0,5.4,2.0,4.0,8.0,6,top400,0.308790,60,高点击率词
9,安佳淡奶油,4920,29258,10877,1.7,153.0,0.0,6.6,0.0,0.0,5.0,19,top400,0.371762,60,高点击率词


In [3]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd

# 设置pandas显示选项以展示更多内容
pd.set_option("display.max_rows", 100)  # 显示最多100行
pd.set_option("display.max_columns", None)  # 显示所有列
pd.set_option("display.width", 1000)  # 设置显示宽度
pd.set_option("display.max_colwidth", 100)  # 设置列最大宽度

import sqlite3


def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    """
    从SLS(Simple Log Service)获取指定日期的用户变体数据。

    Args:
        day (datetime): 要获取数据的日期。
        check_if_local_exist (bool): 是否检查本地数据库中是否已存在数据，默认为True。

    Returns:
        pd.DataFrame: 包含用户变体数据的DataFrame。
    """
    # 构建数据库文件名和表名
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    # 连接到SQLite数据库
    conn = sqlite3.connect(db_file_name)

    # 如果设置为检查本地数据
    if check_if_local_exist:
        try:
            # 尝试从数据库中读取数据
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            # 关闭数据库连接
            conn.close()
            # 返回读取的数据
            return df
        except pd.io.sql.DatabaseError:
            # 如果表不存在，则忽略错误
            pass

    # 构建SLS查询语句
    query = f"""
type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{{digit}}') as api,
    pageName as page_name,
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id,
    type,
    uid,
    date_format(__time__, '%Y%m%d') as ds,
    count(1) as search_times,
    array_join(array_sort(array_agg(distinct regexp_extract(experiment_item, '"variantId":"([^"]+)"', 1))),',') as variant_list
FROM log, 
UNNEST(regexp_extract_all(json_extract_scalar(ai, '$.qh.xm-ab-exp'), '\{{[^}}]+\}}')) as t(experiment_item)
WHERE experiment_item LIKE '%"experimentId":"{EXPERIMENTAL_ID}"%'
GROUP BY 1,2,3,4,5,6
LIMIT 1000000
"""
    # 设置查询的起始时间和结束时间
    print(query)
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    # 从SLS获取数据
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",  # 指定SLS项目
        logstore="xm-mall",  # 指定SLS日志库
        from_time=from_time,  # 指定查询起始时间
        to_time=to_time,  # 指定查询结束时间
    )

    # 将search_times列中的缺失值填充为1，并转换为整数类型
    _df["search_times"] = _df["search_times"].fillna(1).astype(int)
    # 将variant_list列中的缺失值填充为"none"
    _df["variant_list"] = _df["variant_list"].fillna("none")

    # 如果DataFrame不为空
    if not _df.empty:
        # 删除不需要的列
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        # 将数据写入SQLite数据库，如果表已存在则替换
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    # 关闭数据库连接
    conn.close()
    # 返回数据
    return _df


# 创建一个空的DataFrame来存储所有日期的用户变体数据
all_user_variant_df = pd.DataFrame()
# 设置起始日期和结束日期
start_date = datetime.strptime(START_DATE, "%Y-%m-%d")
end_date = datetime.now()
# 从起始日期开始循环，直到结束日期
current_date = start_date
while current_date <= end_date:
    # 检查是否是今天
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    # 如果是今天,则跳过，因为今天的数据可能不完整
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    # 获取当前日期的用户变体数据
    df = get_user_variant_of_date_from_sls(current_date, check_if_local_exist=True)
    # 将当前日期的数据添加到总的DataFrame中
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    # 日期增加一天
    current_date += timedelta(days=1)

# 显示前10行数据
all_user_variant_df.head(10)


type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{digit}') as api,
    pageName as page_name,
    regexp_extract(experiment_item, '"experimentId":"([^"]+)"', 1) as experiment_id,
    type,
    uid,
    date_format(__time__, '%Y%m%d') as ds,
    count(1) as search_times,
    array_join(array_sort(array_agg(distinct regexp_extract(experiment_item, '"variantId":"([^"]+)"', 1))),',') as variant_list
FROM log, 
UNNEST(regexp_extract_all(json_extract_scalar(ai, '$.qh.xm-ab-exp'), '\{[^}]+\}')) as t(experiment_item)
WHERE experiment_item LIKE '%"experimentId":"product_search_custom_scoring"%'
GROUP BY 1,2,3,4,5,6
LIMIT 1000000

即将获取数据: =====> 2025-09-12 00:00:00 2025-09-12 23:59:59.999999 xm-mall: 
type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+',
>=====数条数:8498

type:a and ap:/mall/sku/page and pageName:/search/goods-new
| SELECT 
    regexp_replace(ap, '\d+','{digit}') as api,
    pageName 

,api,page_name,experiment_id,type,uid,ds,search_times,variant_list
0,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,484130,20250912,15,V3
1,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,71841,20250912,4,V2
2,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,209712,20250912,7,V3
3,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,313115,20250912,2,V2
4,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,221276,20250912,2,V1
5,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,229374,20250912,5,V2
6,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,251226,20250912,6,V1
7,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,557175,20250912,8,V1
8,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,432952,20250912,2,V2
9,/mall/sku/page,/search/goods-new,product_search_custom_scoring,a,425420,20250912,2,V1


In [4]:
import pandasql
stats=pandasql.sqldf("""select ds,variant_list,count(distinct uid) unique_user 
                     from all_user_variant_df group by ds,variant_list order by ds desc,variant_list""")

display(stats)

,ds,variant_list,unique_user
0,20250914,V1,1946
1,20250914,V2,1904
2,20250914,V3,1913
3,20250914,V4,1877
4,20250913,V1,1944
5,20250913,V2,1893
6,20250913,V3,1968
7,20250913,V4,1932
8,20250912,V1,2088
9,20250912,V2,2119


In [5]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query = """
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,search_query,type,uid,ds
from(
select uid,date_format(__time__, '%Y%m%d') ds,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""


def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_view_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_view_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

即将获取数据: =====> 2025-09-12 00:00:00 2025-09-12 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:204183
即将获取数据: =====> 2025-09-13 00:00:00 2025-09-13 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:182619
即将获取数据: =====> 2025-09-14 00:00:00 2025-09-14 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods-new |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx
>=====数条数:183633
今天的数据还未完整，跳过:2025-09-15 00:00:00


In [6]:
from sls_client import get_sls_raw_data_by_query

click_query = """
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_item, 'name:([^,]+)', 1)) AS name,
  coalesce(sku,regexp_extract(sku_item, 'sku:([\dA-Z]+)', 1)) AS sku,
  coalesce(pid,regexp_extract(sku_item, 'pid:([^,]+)', 1)) AS pid,
  coalesce(pdid,regexp_extract(sku_item, 'pdid:(\d+)', 1)) AS pdid,bid,
ds,search_query,type,uid,page_name,sku_item,coalesce(linkInfo,url)linkInfo from(
select uid,date_format(__time__, '%Y%m%d') ds,replace(replace(split_part(url_decode(split_part(url,'#/',2)),'?',2),'=',':'),'&',',') url,
bid_list.sku_item,pageName as page_name,bid,idx,name,sku,pid,pdid,linkInfo,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 1000000)"""

# click_query = "type:cl and pageName:/search/goods"


def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_raw_data_by_query(
        query=click_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(
            columns=[
                "__source__",
                "__time__",
                "userAgent",
                "url",
                "__topic__",
                "__tag__:__client_ip__",
                "__tag__:__receive_time__",
                "__time_ns_part__",
            ],
            inplace=True,
            errors="ignore",
        )
        _df["ds"] = day.strftime("%Y%m%d")
        _df.to_sql(table_name, conn, if_exists="replace", index=False)
    conn.close()
    return _df


all_user_sku_click_df = pd.DataFrame()
current_date = start_date
while current_date <= end_date:
    is_today = current_date.strftime("%Y%m%d") == end_date.strftime("%Y%m%d")
    if is_today:
        print(f"今天的数据还未完整，跳过:{current_date}")
        break
    df = get_user_sku_click_of_date_from_sls(current_date, check_if_local_exist=True)
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

即将获取数据: =====>from_time:2025-09-12 00:00:00, to_time:2025-09-12 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:38542
即将获取数据: =====>from_time:2025-09-13 00:00:00, to_time:2025-09-13 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:33535
即将获取数据: =====>from_time:2025-09-14 00:00:00, to_time:2025-09-14 23:59:59.999999, logstore:xm-mall, query:
type:cl and pageName:/search/goods-new |
select   coalesce(idx,regexp_extract(sku_item, 'idx:(\d+)', 1)) AS idx,
  coalesce(name,regexp_extract(sku_i
>=====数据条数:32774
今天的数据还未完整，跳过:2025-09-15 00:00:00


,idx,name,sku,pid,pdid,bid,ds,search_query,type,uid,page_name,sku_item,linkInfo
0,null,美都无籽西瓜 毛重31-34斤/一级/4粒,611260163,加购弹窗,454,"name:美都无籽西瓜 毛重31-34斤/一级/4粒,pid:加购弹窗,sku:611260163,pdid:454,stock:242",20250912,西瓜,cl,532445,/search/goods-new,"name:美都无籽西瓜 毛重31-34斤/一级/4粒,pid:加购弹窗,sku:611260163,pdid:454,stock:242","name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
1,5,芒果（三级果） 净重12-12.5斤/三级/单果约300g+,5450535868,唤起购买,1307,undefined,20250912,芒果,cl,501076,/search/goods-new,undefined,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
2,0,C味三色大芋圆 1KG*10包,661210374451,goods,3092,"idx:0,name:C味三色大芋圆 1KG*10包,pid:goods,sku:661210374451,salePrice:156,pdid:3092,stock:10000,ext:cross",20250912,芋圆,cl,410179,/search/goods-new,"idx:0,name:C味三色大芋圆 1KG*10包,pid:goods,sku:661210374451,salePrice:156,pdid:3092,stock:10000,ext:cross","name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
3,0,泰国无核榴莲冻肉(阿榴哥) 6斤*1包,78083116121,唤起购买,2826,undefined,20250912,榴莲,cl,169866,/search/goods-new,undefined,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
4,null,加入购物车,791427772364,加购弹窗,6870,"name:加入购物车,pid:加购弹窗,sku:791427772364,pdid:6870,stock:527",20250912,糖,cl,19250,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:791427772364,pdid:6870,stock:527","name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
5,9,蒙特瑞草莓 250g*1盒/一级/4*7,611184853,唤起购买,452,undefined,20250912,树莓,cl,448802,/search/goods-new,undefined,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
6,null,加入购物车,611184853,加购弹窗,452,"name:加入购物车,pid:加购弹窗,sku:611184853,pdid:452,stock:22",20250912,树莓,cl,448802,/search/goods-new,"name:加入购物车,pid:加购弹窗,sku:611184853,pdid:452,stock:22","name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
7,0,澄善杂果果馅 5KG*1桶,852323254100,唤起购买,11534,undefined,20250912,澄善杂果果馅,cl,439530,/search/goods-new,undefined,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
8,2,蓝莓盒装 125g*2盒/一级/果径约14-16mm,634778876,唤起购买,560,undefined,20250912,蓝莓,cl,182938,/search/goods-new,undefined,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"
9,0,TS韩国幼砂糖 30KG*1包,T001S01H001,goods,98,"idx:0,name:TS韩国幼砂糖 30KG*1包,pid:goods,sku:T001S01H001,salePrice:250,pdid:98,stock:10000,ext:cross",20250912,糖,cl,19250,/search/goods-new,"idx:0,name:TS韩国幼砂糖 30KG*1包,pid:goods,sku:T001S01H001,salePrice:250,pdid:98,stock:10000,ext:cross","name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no"


In [7]:
import re

all_user_sku_click_explored = []
pattern = re.compile(r'idx:(?P<idx>\d+).*?name:(?P<name>[^,]+).*?pid:(?P<pid>[^,]+).*?sku:(?P<sku>[^,]+).*?pdid:(?P<pdid>[^,]+)')

for index, row in all_user_sku_click_df.iterrows():
    _dict = row.to_dict()
    search_query=_dict["search_query"]
    if not search_query or f"{search_query}" == "":
        search_query = _dict["linkInfo"]
        # 搜索pdName，如果没找到，则search_query为空字符串
        match = re.search(r'pdName:([^,]+)', search_query)
        search_query = match.group(1) if match else ""
    bid = _dict["bid"]
    for bid_item in bid.split(";"):
        sku_info = {}
        sku_info.update(_dict)
        sku_info["search_query"] = search_query
        sku_info["bid"] = bid_item
        if 'undefined' in bid_item:
            all_user_sku_click_explored.append(sku_info)
        else:
            try:
                idx = pdid = sku = pid = name = None
                
                idx_match = re.search(r'idx:(\d+)', bid_item)
                if idx_match:
                    idx = idx_match.group(1)
                    
                pdid_match = re.search(r'pdid:(\d+)', bid_item)
                if pdid_match:
                    pdid = pdid_match.group(1)
                    
                sku_match = re.search(r'sku:([\dA-Za-z]+)', bid_item)
                if sku_match:
                    sku = sku_match.group(1)
                    
                pid_match = re.search(r'pid:([^,]+)', bid_item)
                if pid_match:
                    pid = pid_match.group(1)
                    
                name_match = re.search(r'name:([^,]+)', bid_item)
                if name_match:
                    name = name_match.group(1)
                    
                sku_info.update({
                    "idx": idx,
                    "pdid": pdid, 
                    "sku": sku,
                    "pid": pid,
                    "name": name
                })
                all_user_sku_click_explored.append(sku_info)
            except Exception as e:
                print(e, bid_item)
                raise e

all_user_sku_click_explored_df = pd.DataFrame(all_user_sku_click_explored)
all_user_sku_click_explored_df[['bid','sku','name','idx','pid','pdid','linkInfo','search_query']].head(5)

,bid,sku,name,idx,pid,pdid,linkInfo,search_query
0,"name:美都无籽西瓜 毛重31-34斤/一级/4粒,pid:加购弹窗,sku:611260163,pdid:454,stock:242",611260163,美都无籽西瓜 毛重31-34斤/一级/4粒,None,加购弹窗,454,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no",西瓜
1,undefined,5450535868,芒果（三级果） 净重12-12.5斤/三级/单果约300g+,5,唤起购买,1307,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no",芒果
2,"idx:0,name:C味三色大芋圆 1KG*10包,pid:goods,sku:661210374451,salePrice:156,pdid:3092,stock:10000,ext:cross",661210374451,C味三色大芋圆 1KG*10包,0,goods,3092,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no",芋圆
3,undefined,78083116121,泰国无核榴莲冻肉(阿榴哥) 6斤*1包,0,唤起购买,2826,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no",榴莲
4,"name:加入购物车,pid:加购弹窗,sku:791427772364,pdid:6870,stock:527",791427772364,加入购物车,None,加购弹窗,6870,"name:Search,word:创意月饼配方,linkShadingWord:[object Object],isTiming:no",糖


In [8]:
print(all_user_variant_df.columns)
print(all_user_sku_view_df.columns)
print(all_user_sku_click_df.columns)

all_sku_view_data_df = all_user_sku_view_df[
    [
        "idx",
        "name",
        "sku",
        "pid",
        "pdid",
        "uid",
        "ds",
        "search_query",
        "type",
    ]
].merge(
    all_user_variant_df[["uid", "ds", "variant_list", "search_times"]],
    on=["uid", "ds"],
    how="left",
)


Index(['api', 'page_name', 'experiment_id', 'type', 'uid', 'ds', 'search_times', 'variant_list'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'search_query', 'type', 'uid', 'ds'], dtype='object')
Index(['idx', 'name', 'sku', 'pid', 'pdid', 'bid', 'ds', 'search_query', 'type', 'uid', 'page_name', 'sku_item', 'linkInfo'], dtype='object')


In [9]:
all_user_sku_click_explored_df.groupby("pid").size().reset_index(name="count").sort_values(
    "count", ascending=False
).head(10)

,pid,count
5,加购弹窗,42699
6,唤起购买,37092
2,goods,21905
8,横版筛选栏,1167
7,商品列表,1129
3,mini榜单,733
9,竖版筛选栏,77
0,AI采购,38
1,AI问题,7
4,null,2


In [10]:
user_click_with_variant_df = all_user_sku_click_explored_df.merge(
    all_user_variant_df[["uid", "ds", "variant_list"]],
    on=["uid", "ds"],
    how="left",
)

user_click_with_variant_df["action_type"] = user_click_with_variant_df.apply(
    lambda row: (
        "加入购物车"
        if row["pid"] == "加购弹窗" and row["name"] == "加入购物车"
        else "商品详情" if row["pid"] == "goods" else row["pid"]
    ),
    axis=1,
)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].fillna(-1)
user_click_with_variant_df['idx'] = user_click_with_variant_df['idx'].replace('null', -1).astype(int)

In [11]:
import pandasql

user_click_with_variant_statistics_df = pandasql.sqldf(
"""
select uid,variant_list,ds,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,
count(case when action_type='商品详情' then 1 end) as 商品详情cnt,
count(case when action_type='加入购物车' then 1 end) as 加入购物车cnt,
count(case when action_type='唤起购买' and variant_list is not null then 1 end) as 唤起购买cnt,
count(case when (action_type='唤起购买' and variant_list is not null) or action_type='商品详情' then 1 end) as 总点击cnt,
count(case when (action_type='唤起购买' and variant_list is not null or action_type='商品详情') and idx>=0 and idx<=5 then 1 end) as 首屏总点击cnt,
round(avg(case when action_type='商品详情' or action_type='唤起购买' then idx end),1) as avg点击位置,
coalesce(max(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as max点击位置,
coalesce(min(case when action_type='商品详情' or action_type='唤起购买' then idx end),100) as min点击位置,
count(distinct sku) 点击SKU_cnt,
count(distinct search_query) 搜索词cnt                        
from user_click_with_variant_df a
left join top_query_df b on a.search_query = b.query
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
""")

user_click_with_variant_statistics_df.head(5)

,uid,variant_list,ds,搜索频次标签,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt
0,100027,V3,20250913,top400以外,0,0,1,1,1,0.0,0,0,1,1
1,10008,V3,20250913,top400,1,0,1,2,2,0.0,0,0,2,2
2,100144,V4,20250913,top400,1,0,0,1,1,0.0,0,0,1,1
3,100144,V4,20250913,top400以外,1,0,0,1,1,4.0,4,4,1,1
4,100171,V2,20250913,top400,0,2,2,2,2,0.0,0,0,2,2


In [12]:
# all_sku_view_data_df
# user_click_with_variant_df
# top_query_df

query_level_analytics=f"""
select 
    a.variant_list,a.search_query,c.搜索频次标签,
    count(distinct a.uid) as uv,count(distinct b.uid) as 点击uv,
    count(1) as 查看cnt,count(case when b.uid is not null then 1 end) as 点击cnt
    ,round(1.00 * count(case when b.uid is not null then 1 end)/count(1),4) as 点击率
    ,max(c.searched_users) total_searched_users
from all_sku_view_data_df a
left join user_click_with_variant_df b on a.uid = b.uid and a.ds = b.ds and a.search_query = b.search_query and a.idx = b.idx
left join top_query_df c on a.search_query = c.query
group by a.variant_list,a.search_query,c.搜索频次标签
order by total_searched_users desc,a.variant_list
limit 1600
"""

query_level_analytics_df = pandasql.sqldf(query_level_analytics)
query_level_analytics_df.to_csv(f"./data/搜索AB-查询词粒度分析-top400-{START_DATE}.csv", index=False)
query_level_analytics_df

,variant_list,search_query,搜索频次标签,uv,点击uv,查看cnt,点击cnt,点击率,total_searched_users
0,None,芒果,top400,1,0,1,0,0.0000,12611
1,V1,芒果,top400,624,463,11205,1198,0.1069,12611
2,V2,芒果,top400,638,472,10899,1309,0.1201,12611
3,V3,芒果,top400,655,484,14082,1569,0.1114,12611
4,V4,芒果,top400,653,490,13511,1460,0.1081,12611
...,...,...,...,...,...,...,...,...,...
1595,V2,半熟芝士,top400,3,3,88,23,0.2614,252
1596,V2,杏仁片,top400以外,5,1,68,1,0.0147,252
1597,V2,红心火龙果,top400,7,7,53,8,0.1509,252
1598,V2,肠,top400,5,1,107,1,0.0093,252


In [13]:
null_search_query_df = pandasql.sqldf(
    """select case when search_query is null or search_query = 'null' then 'null-search-query' else 'normal' end has_search_query,
                                    count(1) cnt from all_sku_view_data_df group by 1"""
)
null_search_query_df
# 约有2.3%的数据没有搜索词，这部分需要过滤掉
all_sku_view_data_df = all_sku_view_data_df[
    all_sku_view_data_df["search_query"] != "null"
]

In [14]:
all_sku_view_data_df["idx"] = all_sku_view_data_df["idx"].fillna(-1).astype(int)

# 用来单独过滤某些词的表现
query_list=['牛奶','柠檬']
query_list_str='","'.join(query_list)


user_view_with_variant_statistics_df = pandasql.sqldf(
f"""
select uid,ds,variant_list,coalesce(b.搜索频次标签,'其他') as 搜索频次标签,min(a.search_query) sample_query,
count(1) as 商品查看cnt,
count(distinct sku) as 查看SKU_cnt,
count(distinct search_query) as 查看搜索词cnt,
max(idx) as max查看位置,
max(search_times) as 搜索翻页数cnt
from all_sku_view_data_df a
left join top_query_df b on a.search_query = b.query
-- where a.search_query in ("{query_list_str}")
group by uid,variant_list,ds,coalesce(b.搜索频次标签,'其他')
"""
)

print("unique sample_query:", user_view_with_variant_statistics_df['sample_query'].unique(), len(user_view_with_variant_statistics_df['sample_query'].unique()))

unique sample_query: ['安佳奶油' '草莓' '无花果' ... '西柚酱' '千岛酱' '果蔬汁'] 4268


In [15]:
first_view_sql=f"""
select variant_list,uid,search_query,count(*) as sku_viewed_cnt
from all_sku_view_data_df
where cast(idx as bigint)<=5
group by variant_list,uid,search_query
order by sku_viewed_cnt desc
"""

first_view_sql_df = pandasql.sqldf(first_view_sql)
first_view_sql_df



,variant_list,uid,search_query,sku_viewed_cnt
0,V3,530482,大成早餐腿排(奥尔良味),68
1,V3,287729,芒果,54
2,V3,482352,芒果,53
3,V2,554961,蓝莓,52
4,V3,437486,安佳,52
...,...,...,...,...
51163,V4,84578,白爆花,1
51164,V4,84685,铁塔,1
51165,V4,85025,块状,1
51166,V4,88911,铁塔,1


In [16]:
first_view_click_sql=f"""
select uid,search_query,count(*) as sku_clicked_cnt,min(idx),max(idx),avg(idx)
from all_user_sku_click_df
where idx is not null and idx != 'null' and cast(idx as bigint)<=5
group by uid,search_query
order by sku_clicked_cnt desc
"""

first_view_click_sql_df = pandasql.sqldf(first_view_click_sql)
first_view_click_sql_df



,uid,search_query,sku_clicked_cnt,min(idx),max(idx),avg(idx)
0,264766,羽衣,30,0,0,0.000000
1,373476,芒果,21,2,3,2.523810
2,44065,妞丝达特奥利奥半熟芝士,21,0,3,1.095238
3,556625,羽衣甘蓝,20,0,0,0.000000
4,567918,芒果,19,0,5,2.315789
...,...,...,...,...,...,...
30990,99863,树莓,1,0,0,0.000000
30991,99888,铁塔,1,0,0,0.000000
30992,99999,咖啡豆,1,1,1,1.000000
30993,99999,果蔬汁,1,1,1,1.000000


In [17]:
first_view_merged_df=first_view_sql_df.merge(first_view_click_sql_df, on=["uid", "search_query"], how="left")
first_view_merged_df.head(5)

grouped_first_view_df=first_view_merged_df.groupby("search_query").agg({"sku_viewed_cnt": "sum", "sku_clicked_cnt": "sum"}).reset_index()
grouped_first_view_df.columns=["search_query","总浏览量","总点击量"]
grouped_first_view_df.head(5)

first_view_merged_df=first_view_merged_df.merge(grouped_first_view_df, on="search_query", how="left")
first_view_merged_df=first_view_merged_df.sort_values("总点击量", ascending=False)
first_view_merged_df["是否实验组"]=first_view_merged_df.apply(lambda x: "对照组" if x["variant_list"] in ["V1","V2"] else "实验组", axis=1)
first_view_merged_df.head(5)

first_view_merged_stats=pandasql.sqldf("""
select search_query,是否实验组, sum(sku_viewed_cnt) as 浏览量, sum(sku_clicked_cnt) as 点击量
    ,sum(sku_clicked_cnt)*1.00/sum(sku_viewed_cnt) as 点击率
    ,min(总浏览量) as 词的总浏览量
from first_view_merged_df 
group by 是否实验组,search_query
order by min(总浏览量) desc
""")

first_view_merged_stats.to_csv(f"./data/搜索AB-首屏点击分析-{START_DATE}.csv", index=False)
first_view_merged_stats.head(40)

,search_query,是否实验组,浏览量,点击量,点击率,词的总浏览量
0,芒果,实验组,10062,996.0,0.098986,18847
1,芒果,对照组,8785,1425.0,0.162208,18847
2,蓝莓,实验组,5366,978.0,0.182259,10167
3,蓝莓,对照组,4801,844.0,0.175797,10167
4,牛奶,实验组,3698,592.0,0.160087,7210
5,牛奶,对照组,3512,587.0,0.167141,7210
6,草莓,实验组,3437,706.0,0.205412,6817
7,草莓,对照组,3380,722.0,0.213609,6817
8,柠檬,实验组,3065,501.0,0.163458,6141
9,柠檬,对照组,3076,494.0,0.160598,6141


In [18]:
controled_group_df=first_view_merged_stats[first_view_merged_stats["是否实验组"]=="对照组"]
controled_group_dict={}
for row_index, row in controled_group_df.iterrows():
    controled_group_dict[row["search_query"]]=row["点击率"]

first_view_merged_stats["点击率提升"]=first_view_merged_stats.apply(lambda row: 100.00*(row["点击率"]/controled_group_dict.get(row["search_query"],0.01) - 1), axis=1)
first_view_merged_stats["点击率提升"]=first_view_merged_stats["点击率提升"].round(2)
first_view_merged_stats.to_csv(f"./data/搜索AB-首屏点击分析-点击率提升-{START_DATE}.csv",index=False)
first_view_merged_stats.head(50)

,search_query,是否实验组,浏览量,点击量,点击率,词的总浏览量,点击率提升
0,芒果,实验组,10062,996.0,0.098986,18847,-38.98
1,芒果,对照组,8785,1425.0,0.162208,18847,0.00
2,蓝莓,实验组,5366,978.0,0.182259,10167,3.68
3,蓝莓,对照组,4801,844.0,0.175797,10167,0.00
4,牛奶,实验组,3698,592.0,0.160087,7210,-4.22
5,牛奶,对照组,3512,587.0,0.167141,7210,0.00
6,草莓,实验组,3437,706.0,0.205412,6817,-3.84
7,草莓,对照组,3380,722.0,0.213609,6817,0.00
8,柠檬,实验组,3065,501.0,0.163458,6141,1.78
9,柠檬,对照组,3076,494.0,0.160598,6141,0.00


In [19]:
first_view_merged_total_stats=pandasql.sqldf("""
select 是否实验组, sum(sku_viewed_cnt) as 浏览量, sum(sku_clicked_cnt) as 点击量
    ,sum(sku_clicked_cnt)*1.00/sum(sku_viewed_cnt) as 点击率
from first_view_merged_df 
group by 是否实验组
""")
first_view_merged_total_stats

,是否实验组,浏览量,点击量,点击率
0,实验组,142586,23338.0,0.163677
1,对照组,142237,23440.0,0.164795


In [20]:
all_data_df = user_view_with_variant_statistics_df.merge(
    user_click_with_variant_statistics_df, on=["uid", "ds", "variant_list","搜索频次标签"], how="left"
)
# 定义计数列名列表
count_columns = ["商品详情cnt", "加入购物车cnt", "唤起购买cnt", "首屏总点击cnt", "总点击cnt"]
# 遍历计数列
for col in count_columns:
    # 将空值填充为0并转换为整数类型
    all_data_df[col] = all_data_df[col].fillna(0).astype(int)

# 创建 "用户是否点击" 列
all_data_df["用户是否点击"] = (all_data_df["总点击cnt"] > 0).astype(int)

# 定义费率计算相关列名列表
rate_columns = [
    ("sku_click_rate", "商品详情cnt", "商品查看cnt"), # 商品详情点击率
    ("add_cart_rate", "加入购物车cnt", "商品查看cnt"), # 加入购物车率
    ("popup_click_rate", "唤起购买cnt", "商品查看cnt"), # 唤起购买率
]

# 遍历费率列
for rate_col, num_col, den_col in rate_columns:
    # 计算费率，空值填充0，保留5位小数，转换为浮点数
    all_data_df[rate_col] = (all_data_df[num_col] / all_data_df[den_col]).fillna(0).round(5).astype(float)

all_data_df.head(5)

,uid,ds,variant_list,搜索频次标签,sample_query,商品查看cnt,查看SKU_cnt,查看搜索词cnt,max查看位置,搜索翻页数cnt,商品详情cnt,加入购物车cnt,唤起购买cnt,总点击cnt,首屏总点击cnt,avg点击位置,max点击位置,min点击位置,点击SKU_cnt,搜索词cnt,用户是否点击,sku_click_rate,add_cart_rate,popup_click_rate
0,,20250912,None,top400,安佳奶油,4,4,1,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.00000
1,,20250913,None,top400,草莓,4,4,1,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.00000
2,,20250914,None,top400,无花果,32,14,3,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.00000
3,,20250914,None,top400以外,薄荷糖浆,4,4,1,3,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,0.00000
4,100027,20250913,V3,top400以外,金钻清恬植脂奶油,3,3,1,2,1.0,0,0,1,1,1,0.0,0.0,0.0,1.0,1.0,1,0.0,0.0,0.33333


In [21]:
from IPython.core.display import HTML
import pandas as pd

css = """
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.0.0/dist/css/bootstrap.min.css" integrity="sha384-Gn5384xqQ1aoWXA+058RXPxPg6fy4IWvTNh0E263XmFcJlSAwiGgFAW/dAiS6JXm" crossorigin="anonymous">
<style type=\"text/css\">
#abTesting table,#abTesting .table {
    color: #333;
    font-family: unset;
    font-size: 12px;
    line-height: 1.5;
    width: 95vw;
    border-collapse:
    collapse; 
    border-spacing: 0;
    font-family: "SF Pro SC", "SF Pro Text", "SF Pro Icons", "PingFang SC", "Helvetica Neue", "Helvetica", "Arial", sans-serif;
}

body{
    padding-left: 1rem;
    padding-top: 1vh;
}

tr{
    border-bottom: 1px solid #C1C3D1;
}

tr:nth-child(even) {
    background-color: #F8F8F8;
}

#abTesting td, #abTesting th {
    /* border: 1px solid transparent; No more visible border */
    height: 30px;
    padding: 0.2rem;
}

#abTesting table tbody td,#abTesting .table tbody td{
    padding: 0.1rem .75rem;
    vertical-align: middle;
}

th {
    background-color: #DFDFDF; /* Darken header a bit */
    font-weight: bolder;
    font-size: larger;
    color: #000;
    text-align: center;
}
</style>
"""


def display_p_value_below_005(row: pd.Series, p_value_col_name: str = "p_value"):
    p_value = row[p_value_col_name]
    color = "black"
    if p_value is not None and p_value <= 0.05:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{p_value}</span>"""


def display_diff_to_v2(row: pd.Series, metric: str = "diff_to_v2%"):
    diff = row[metric]
    color = "green"
    if diff is not None and float(diff) > 0.0:
        color = "red"
    return f"""<span style='font-weight:bolder;color:{color};'>{diff:.4f} %</span>"""


def dataframe_to_html(df: pd.DataFrame, title: str):
    df_to_display = df.copy()

    df_to_display["p_value"] = df_to_display.apply(display_p_value_below_005, axis=1)
    df_to_display["diff_to_v2%"] = df_to_display.apply(display_diff_to_v2, axis=1)

    html_df = df_to_display.to_html(
        escape=False, index=False, classes="table dataframe"
    )
    html_content = f"""<html><head><meta charset="UTF-8">
    <meta name="title" content="{title}">
    {css}
    </head><body>
    <h2>{title}</h2>
    <h4>当P-value <= 0.05时表示实验结果统计学显著</h4>
    <span>统计学显著时，既可能表示该试验组是好于对照组，也可能是坏于对照组</span>
    <div id="abTesting">{html_df}</div></body></html>"""

    return html_content

In [22]:
import pandas as pd
from scipy.stats import ttest_ind


def calculate_p_values(
    df: pd.DataFrame,
    metric: str = "商品详情cnt",
    control_variant: str = "V2",
) -> pd.DataFrame:
    """
    Calculate p-values for each combination of category1 and page_name.
    Compares metric between control group (V1) and each of V2, V3, V4.

    Parameters:
    - df (pd.DataFrame): The input DataFrame containing A/B test data.
    - metric (str): The metric column to be analyzed (default is 'added_quantity').

    Returns:
    - pd.DataFrame: A DataFrame with category1, page_name, variant, p-value, and statistics columns.
    """
    p_values = []

    control = df[df["variant_list"] == control_variant][metric]
    control_avg = control.mean()

    for variant in ["V1", "V2", "V3", "V4"]:
        test_group = df[df["variant_list"] == variant]
        test = test_group[metric]

        # print(
        #     f'variant:{variant}, test_group ds length: {len(test_group["ds"].unique())}'
        # )
        if len(test_group["ds"].unique()) <= 0:
            continue

        # Calculate statistics
        stats = {
            "均值": round(test.mean(), 4),
            "std": round(test.std(), 4),
            f"diff_to_{control_variant}%".lower(): round(
                100.00 * (test.mean() - control_avg) / control_avg, 2
            ),
            "q50": test.quantile(0.5),
            "q75": test.quantile(0.75),
            "q90": test.quantile(0.9),
            "q95": test.quantile(0.95),
            "q97": test.quantile(0.97),
            "q99": test.quantile(0.99),
            "q995": test.quantile(0.995),
            "max": test.max(),
            "日均总数": round(test.sum() / len(test_group["ds"].unique())),
            "日均实验UV": round(
                len(test_group[["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日均转化UV": round(
                len(test_group[test_group[metric] > 0][["uid", "ds"]].drop_duplicates())
                / len(test_group["ds"].unique())
            ),
            "日期范围": f"{test_group['ds'].min()}~{test_group['ds'].max()}".replace(
                "2025", ""
            ),
            "metric": metric,
        }

        # Ensure both groups have enough data for a valid t-test
        if len(control) > 1 and len(test) > 1:
            # Perform independent t-test
            stat, p_val = ttest_ind(control, test, equal_var=False, nan_policy="omit")
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": round(p_val, 4),
                    **stats,
                }
            )
            # print(f"stat:{stat}")
        else:
            # Not enough data for statistical testing
            p_values.append(
                {
                    "variant_list": variant,
                    "p_value": None,
                    **stats,
                }
            )

    return pd.DataFrame(p_values)

In [23]:
# Define the desired order for sorting
variant_order = ["V1", "V2", "V3", "V4"]


# Create a custom sort key function
def sort_key(variant):
    # Split variant by commas
    parts = variant.split(",")
    # Determine the order based on the first variant in the list
    if parts[0] in variant_order:
        return variant_order.index(parts[0])
    else:
        return len(variant_order)  # Place all other variants after V1, V2, V3, V4


metrics_list = [
    "sku_click_rate",
    "avg点击位置",
    "唤起购买cnt",
    "总点击cnt",
    "商品详情cnt",
    "加入购物车cnt",
    "商品查看cnt",
]
all_p_values_df = pd.DataFrame()
for metric in metrics_list:
    all_p_values_of_same_metric_df = pd.DataFrame()
    for label, group_df in all_data_df.groupby("搜索频次标签"):
        if label == "其他":
            # print("ignore 其他")
            continue
        p_values_df = calculate_p_values(
            group_df,
            metric=metric,
        )

        p_values_df["搜索频次"] = label

        p_values_df["variant_list"] = p_values_df["variant_list"].apply(
            lambda x: x if x in variant_order else "X_" + x
        )
        p_values_df = p_values_df.sort_values(
            by="variant_list", key=lambda x: x.map(sort_key)
        )
        p_values_df["variant_list"] = p_values_df["variant_list"].str.replace("X_", "")
        all_p_values_of_same_metric_df = pd.concat(
            [all_p_values_of_same_metric_df, p_values_df], ignore_index=True
        )
        all_p_values_df = pd.concat([all_p_values_df, p_values_df], ignore_index=True)

    title = f"搜索AB--{metric}_p-value分布-{p_values_df.iloc[0]['日期范围']}"

    html_content = dataframe_to_html(df=all_p_values_of_same_metric_df, title=title)
    file_path = f"./data/{title}.html"

    # 保存HTML到本地文件：
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"写入HTML成功！{file_path}")


all_p_values_df = all_p_values_df[['metric', '搜索频次', 'variant_list', 'p_value', '均值', 'std', 'diff_to_v2%', 
         'q50', 'q75', 'q90', 'q95', 'q97', 'q99', 'q995', 'max', '日均总数', 
         '日均实验UV', '日均转化UV', '日期范围']]

title_all = f"搜索AB--指标全集_p-value分布-{all_p_values_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_p_values_df, title=title_all)
file_path = f"./data/{title_all}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
# 根据你的截图，完整的列顺序调整为：
all_p_values_df

写入HTML成功！./data/搜索AB--sku_click_rate_p-value分布-0912~0914.html
写入HTML成功！./data/搜索AB--avg点击位置_p-value分布-0912~0914.html
写入HTML成功！./data/搜索AB--唤起购买cnt_p-value分布-0912~0914.html
写入HTML成功！./data/搜索AB--总点击cnt_p-value分布-0912~0914.html
写入HTML成功！./data/搜索AB--商品详情cnt_p-value分布-0912~0914.html
写入HTML成功！./data/搜索AB--加入购物车cnt_p-value分布-0912~0914.html
写入HTML成功！./data/搜索AB--商品查看cnt_p-value分布-0912~0914.html
写入HTML成功！./data/搜索AB--指标全集_p-value分布-0912~0914.html


,metric,搜索频次,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围
0,sku_click_rate,top400,V1,0.6559,0.0529,0.1093,-1.82,0.0,0.058820,0.181820,0.25,0.33333,0.500,0.600000,1.50000,92,1738,689,0912~0914
1,sku_click_rate,top400,V2,1.0000,0.0539,0.1140,0.00,0.0,0.058820,0.179156,0.25,0.33333,0.500,0.666670,1.33333,91,1693,667,0912~0914
2,sku_click_rate,top400,V3,0.9162,0.0536,0.1055,-0.42,0.0,0.062500,0.200000,0.25,0.33333,0.500,0.571430,1.50000,94,1750,699,0912~0914
3,sku_click_rate,top400,V4,0.6569,0.0529,0.1128,-1.86,0.0,0.060915,0.181820,0.25,0.33333,0.500,0.608459,3.00000,90,1697,660,0912~0914
4,sku_click_rate,top400以外,V1,0.3432,0.0550,0.1165,6.17,0.0,0.063510,0.190212,0.25,0.33333,0.500,0.680480,1.33333,39,714,263,0912~0914
5,sku_click_rate,top400以外,V2,1.0000,0.0518,0.1047,0.00,0.0,0.058820,0.200000,0.25,0.33333,0.500,0.545450,1.00000,38,726,258,0912~0914
6,sku_click_rate,top400以外,V3,0.2734,0.0556,0.1219,7.24,0.0,0.063005,0.200000,0.25,0.33333,0.500,0.621625,2.33333,41,743,262,0912~0914
7,sku_click_rate,top400以外,V4,0.1556,0.0566,0.1163,9.23,0.0,0.062500,0.229230,0.25,0.33333,0.500,0.666670,1.00000,41,716,254,0912~0914
8,avg点击位置,top400,V1,0.6670,3.5392,7.0735,-1.90,1.0,4.000000,9.320000,15.52,20.00000,29.000,39.110000,190.00000,4930,1738,1018,0912~0914
9,avg点击位置,top400,V2,1.0000,3.6079,7.4470,0.00,1.2,3.700000,8.700000,15.00,20.77300,38.991,49.000000,145.80000,4936,1693,999,0912~0914


In [24]:
# 导入odps_client库中的两个函数：get_odps_sql_result_as_df 用于执行SQL查询并将结果作为DataFrame返回, write_pandas_df_into_odps 用于将pandas DataFrame写入ODPS表
from odps_client import get_odps_sql_result_as_df, write_pandas_df_into_odps

# 定义分区规范字符串，使用当前日期（年-月-日格式）作为分区值
partition_spec = f"pt={datetime.now().strftime('%Y%m%d')}"

# 将DataFrame写入ODPS表
write_pandas_df_into_odps(
    df=all_user_variant_df,  # 要写入的DataFrame，这里是all_user_variant_df，包含了所有用户的变体信息
    table_name="summerfarm_ds.temp_search_ab_all_data_df",  # ODPS表名
    partition_spec=partition_spec,  # 分区规范
    overwrite=True,  # 如果表或分区已存在，是否覆盖
    lifecycle=30,  # 设置表的生命周期为30天
)

# 只分析哪些进入过搜索页面的用户的订单转化结果

# 将开始日期格式化为字符串（年-月-日）
start_date_str = start_date.strftime("%Y-%m-%d")

# 定义SQL查询字符串，用于获取用户订单数据.
# 这段SQL的目的是：从订单表和用户分流表中，根据用户ID和日期进行关联，
# 统计每个用户在不同实验变体下的订单总金额、订单数量和平均订单金额。
order_query = f"""
with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '{start_date_str} 00:00:00'
    AND     m_size = '单店'
),user_variants as (
    select ds as event_date,uid,variant_list
    from summerfarm_ds.temp_search_ab_all_data_df
    where pt=max_pt('summerfarm_ds.temp_search_ab_all_data_df')
)
select a.event_date,a.uid,a.variant_list,sum(b.total_price) as order_gmv,
    count(b.order_no) as order_cnt,round(sum(b.total_price)/count(b.order_no),2) as avg_order_gmv
from user_variants a
left join user_orders b on a.uid=b.m_id and a.event_date=b.order_date
group by a.event_date,a.uid,a.variant_list
"""

# 执行SQL查询并将结果作为DataFrame返回
user_orders_df = get_odps_sql_result_as_df(order_query)
# 显示DataFrame的前两行
user_orders_df.head(2)

2025-09-15 14:17:27 - INFO - DaraFrame字段合集:experiment_id,type,create_time,ds,uid,variant_list,page_name,api_list,pt,search_times,page_ame,api
2025-09-15 14:17:31 - INFO - Tunnel session created: <TableUploadSession id=202509151417316930f60b2485fa99 project=summerfarm_ds table=temp_search_ab_all_data_df partition_spec=pt=20250915>
2025-09-15 14:17:34 - INFO - 成功写入odps:summerfarm_ds.temp_search_ab_all_data_df, partition_spec:pt=20250915, attemp:0
2025-09-15 14:17:41 - INFO - Tunnel session created: <InstanceDownloadSession id=20250915141740991b481a02064b86 project_name=summerfarm_ds instance_id=20250915061734918gi29uv451gk4>
2025-09-15 14:17:41 - INFO - sql:

with user_orders as (
    SELECT  m_id
        ,total_price
        ,order_no
        ,DATE_FORMAT(order_time,'yyyyMMdd') as order_date
    FROM    summerfarm_tech.ods_orders_df
    WHERE   ds = MAX_PT('summerfarm_tech.ods_orders_df')
    AND     status IN (2,3,6)
    AND     order_time >= '2025-09-12 00:00:00'
    AND     m_size = 

,event_date,uid,variant_list,order_gmv,order_cnt,avg_order_gmv
0,20250912,100277,V1,476,1,476
1,20250912,100612,V2,189,2,94.5


In [25]:
user_orders_df["order_gmv"]=user_orders_df["order_gmv"].astype(float)
user_orders_df["avg_order_gmv"]=user_orders_df["avg_order_gmv"].astype(float)
user_orders_df["order_cnt"]=user_orders_df["order_cnt"].astype(int)
user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].describe()

count     13471.000000
mean        590.569850
std        1446.388216
min           7.500000
25%         176.000000
50%         314.500000
75%         611.500000
max      103004.000000
Name: order_gmv, dtype: float64

In [26]:
print(
    f"所有订单的分布:\n",
    user_orders_df[user_orders_df["order_gmv"]>0]["order_gmv"].quantile(
        [0.5, 0.75, 0.95, 0.99, 0.995, 0.996, 0.997, 0.999, 1]
    ),
)


# 这里排除哪些高单价的订单，否则对于数据分析来说不好处理。
user_orders_below_6k_df = user_orders_df[user_orders_df["order_gmv"] <= 6000]
print(
    "排除高单价的订单后的分布:\n",
    user_orders_below_6k_df["order_gmv"].quantile(
        [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ),
)

所有订单的分布:
 0.500       314.5000
0.750       611.5000
0.950      1664.3500
0.990      4660.3000
0.995      6599.2750
0.996      8167.4880
0.997      9318.6832
0.999     15186.2600
1.000    103004.0000
Name: order_gmv, dtype: float64
排除高单价的订单后的分布:
 0.01      48.6298
0.05      87.0000
0.25     176.0000
0.50     312.0000
0.75     605.0000
0.95    1601.5000
0.99    3300.0000
Name: order_gmv, dtype: float64


In [27]:
user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
user_orders_below_6k_df["order_gmv"] = user_orders_below_6k_df["order_gmv"].astype(
    float
)

user_orders_below_6k_df["avg_order_gmv"].fillna(0.0, inplace=True)
user_orders_below_6k_df["avg_order_gmv"] = user_orders_below_6k_df[
    "avg_order_gmv"
].astype(float)

user_orders_below_6k_df["category1"] = "ignore"
user_orders_below_6k_df["page_name"] = "ignore"

user_orders_during_ab_df = user_orders_below_6k_df[
    user_orders_below_6k_df["variant_list"].isin(["V1", "V2", "V3", "V4"])
]
user_orders_during_ab_df.rename(columns={"event_date": "ds"}, inplace=True)

all_order_pvalue_df = pd.DataFrame()
for metric in ["order_gmv", "avg_order_gmv", "order_cnt"]:
    gmv_df = calculate_p_values(df=user_orders_during_ab_df, metric=metric)
    display(gmv_df)
    all_order_pvalue_df = pd.concat([all_order_pvalue_df, gmv_df], ignore_index=True)

title = f"搜索AB--订单转化p-value分布-{all_order_pvalue_df.iloc[0]['日期范围']}"
html_content = dataframe_to_html(df=all_order_pvalue_df, title=title)
file_path = f"./data/{title}.html"

# 保存HTML到本地文件：
with open(file_path, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"写入HTML成功！{file_path}")
display(all_order_pvalue_df)

/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_23861/858576852.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  user_orders_below_6k_df["order_gmv"].fillna(0, inplace=True)
/var/folders/b3/9hcz86fx1_z_8m4121xwbs2h0000gn/T/ipykernel_23861/858576852.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_orders_below_6k_df["order

,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1227,538.6089,650.0898,4.63,330.0,623.00,1110.00,1650.000,2165.19,3687.645,4660.145,5676.0,605396,1124,1124,0912~0914,order_gmv
1,V2,1.0000,514.7593,618.7753,0.00,307.0,596.00,1095.00,1562.250,2077.94,3156.800,4524.230,5555.0,579619,1126,1126,0912~0914,order_gmv
2,V3,0.5295,524.2233,618.5069,1.84,314.7,604.25,1097.15,1640.025,1952.60,3286.510,4500.000,5841.0,590625,1127,1127,0912~0914,order_gmv
3,V4,0.6888,508.7450,605.1151,-1.17,303.5,600.00,1055.60,1574.600,1965.04,3231.628,4322.560,5787.0,553684,1088,1088,0912~0914,order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.2033,429.7931,472.6256,3.55,275.47,523.4375,900.00,1188.3500,1568.245,2475.030,2978.625,5676.0,483087,1124,1124,0912~0914,avg_order_gmv
1,V2,1.0000,415.0667,478.3516,0.00,255.00,506.0000,873.80,1200.0000,1553.000,2405.014,2850.615,5555.0,467365,1126,1126,0912~0914,avg_order_gmv
2,V3,0.8764,416.8635,470.9025,0.43,269.61,520.0000,881.06,1169.9645,1495.945,2211.888,2863.125,5200.0,469666,1127,1127,0912~0914,avg_order_gmv
3,V4,0.7226,411.0316,448.0424,-0.97,257.50,518.0000,869.60,1198.4000,1503.880,2219.240,2675.460,5200.0,447339,1088,1088,0912~0914,avg_order_gmv


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.4151,1.2773,0.7192,1.05,1.0,1.0,2.0,2.0,3.0,4.0,4.0,13,1436,1124,1124,0912~0914,order_cnt
1,V2,1.0000,1.2641,0.6090,0.00,1.0,1.0,2.0,2.0,3.0,4.0,4.0,7,1423,1126,1126,0912~0914,order_cnt
2,V3,0.1069,1.2899,0.7068,2.05,1.0,1.0,2.0,3.0,3.0,4.0,5.0,11,1453,1127,1127,0912~0914,order_cnt
3,V4,0.6233,1.2567,0.6186,-0.59,1.0,1.0,2.0,2.0,3.0,4.0,5.0,6,1368,1088,1088,0912~0914,order_cnt


写入HTML成功！./data/搜索AB--订单转化p-value分布-0912~0914.html


,variant_list,p_value,均值,std,diff_to_v2%,q50,q75,q90,q95,q97,q99,q995,max,日均总数,日均实验UV,日均转化UV,日期范围,metric
0,V1,0.1227,538.6089,650.0898,4.63,330.00,623.0000,1110.00,1650.0000,2165.190,3687.645,4660.145,5676.0,605396,1124,1124,0912~0914,order_gmv
1,V2,1.0000,514.7593,618.7753,0.00,307.00,596.0000,1095.00,1562.2500,2077.940,3156.800,4524.230,5555.0,579619,1126,1126,0912~0914,order_gmv
2,V3,0.5295,524.2233,618.5069,1.84,314.70,604.2500,1097.15,1640.0250,1952.600,3286.510,4500.000,5841.0,590625,1127,1127,0912~0914,order_gmv
3,V4,0.6888,508.7450,605.1151,-1.17,303.50,600.0000,1055.60,1574.6000,1965.040,3231.628,4322.560,5787.0,553684,1088,1088,0912~0914,order_gmv
4,V1,0.2033,429.7931,472.6256,3.55,275.47,523.4375,900.00,1188.3500,1568.245,2475.030,2978.625,5676.0,483087,1124,1124,0912~0914,avg_order_gmv
5,V2,1.0000,415.0667,478.3516,0.00,255.00,506.0000,873.80,1200.0000,1553.000,2405.014,2850.615,5555.0,467365,1126,1126,0912~0914,avg_order_gmv
6,V3,0.8764,416.8635,470.9025,0.43,269.61,520.0000,881.06,1169.9645,1495.945,2211.888,2863.125,5200.0,469666,1127,1127,0912~0914,avg_order_gmv
7,V4,0.7226,411.0316,448.0424,-0.97,257.50,518.0000,869.60,1198.4000,1503.880,2219.240,2675.460,5200.0,447339,1088,1088,0912~0914,avg_order_gmv
8,V1,0.4151,1.2773,0.7192,1.05,1.00,1.0000,2.00,2.0000,3.000,4.000,4.000,13.0,1436,1124,1124,0912~0914,order_cnt
9,V2,1.0000,1.2641,0.6090,0.00,1.00,1.0000,2.00,2.0000,3.000,4.000,4.000,7.0,1423,1126,1126,0912~0914,order_cnt


In [28]:
all_p_values_df.to_csv(
    f"./data/搜索AB--所有指标p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)
all_order_pvalue_df.to_csv(
    f"./data/搜索AB--订单转化p-value分布-{all_p_values_df.iloc[0]['日期范围']}.csv",
    index=False,
)